<a href="https://colab.research.google.com/github/malkunn/Hourly-River-Streamflow-Forecasting-Hackathon./blob/main/StreamFlow_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# 1. LOAD YOUR DATA
# Replace this with your actual competition csv loading
df_train  = pd.concat([pd.read_csv('trainTrackA.csv'), pd.read_csv('trainTrackB.csv')])
df_test   = pd.concat([pd.read_csv('testTrackA.csv'), pd.read_csv('testTrackB.csv')])

# --- FOR DEMO PURPOSES: Creating dummy data ---
data_length = 1000
time = np.linspace(0, 100, data_length)
streamflow = np.sin(time) + np.random.normal(0, 0.1, data_length)
df = pd.DataFrame({'q': streamflow, 'precip': np.abs(np.cos(time))})
# ----------------------------------------------

# 2. PREPROCESSING
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

def create_windows(data, window_size, forecast_size):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_size):
        X.append(data[i : i + window_size])
        # We target the 'streamflow' column (index 0) for the next 4 hours
        y.append(data[i + window_size : i + window_size + forecast_size, 0])
    return np.array(X), np.array(y)

# Use 24 hours of history to predict 4 hours ahead
WINDOW_SIZE = 24
FORECAST_SIZE = 4
X, y = create_windows(scaled_data, WINDOW_SIZE, FORECAST_SIZE)

# 3. TRAIN/TEST SPLIT (Chronological)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 4. BUILD THE BASELINE LSTM
model = Sequential([
    # Input shape is (number of hours back, number of features)
    LSTM(64, activation='tanh', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.2), # Prevents overfitting
    Dense(32, activation='relu'),
    Dense(FORECAST_SIZE) # Output layer: 4 neurons for Q(t) to Q(t+3)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 5. TRAIN
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# 6. EVALUATE
baseline_mse = model.evaluate(X_test, y_test)[0]
print(f"Baseline RMSE: {np.sqrt(baseline_mse)}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.1243 - mae: 0.2775 - val_loss: 0.0536 - val_mae: 0.1978
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0393 - mae: 0.1640 - val_loss: 0.0122 - val_mae: 0.0923
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0131 - mae: 0.0889 - val_loss: 0.0034 - val_mae: 0.0458
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0084 - mae: 0.0703 - val_loss: 0.0023 - val_mae: 0.0385
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0069 - mae: 0.0641 - val_loss: 0.0022 - val_mae: 0.0376
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0069 - mae: 0.0631 - val_loss: 0.0031 - val_mae: 0.0453
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0065 - mae: 0.0622 - val_loss: 0.0033 - val_mae: 0.0468
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0064 - mae: 0.0611 - val_loss: 0.0026 - val_mae: 0.0417
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0055 